# Trabajo de Laboratorio N° 2 — Agente Inteligente con RAG
**Inteligencia Artificial – Sistemas Inteligentes (FCEFyN – UNC)** · Temas: NLP, representaciones vectoriales, LLMs, RAG y agentes

* **Dominio técnico:** documentación de la **API REST de GitHub** (versión `2022-11-28`).
* **Objetivo:** un agente basado en un LLM local que responde preguntas técnicas fundamentadas en un corpus (RAG) y que además puede **consultar datos en vivo** de repositorios reales y **calcular el consumo de cuota** de la API.

| Componente | Elección |
| :--- | :--- |
| Corpus | 4 documentos Markdown (`corpus/`) |
| Segmentación | Una sección (endpoint) por fragmento, por encabezados Markdown; subdivisión recursiva sólo si supera 1500 caracteres |
| Embeddings | `intfloat/multilingual-e5-small` (multilingüe, 384 dimensiones) |
| Base vectorial | FAISS (`IndexFlatIP`, similitud coseno sobre vectores normalizados) |
| Búsqueda | Híbrida: semántica (FAISS) + léxica (BM25), fusionadas con *Reciprocal Rank Fusion*; top-4 |
| LLM | `qwen2.5:7b` ejecutado localmente con **Ollama** |
| Orquestación | LangChain (`create_agent`, ciclo ReAct con *tool calling*) |

**Requisitos previos:** Ollama en ejecución con el modelo descargado (`ollama pull qwen2.5:7b`) y las dependencias de `requirements.txt` instaladas. Ver `README.md`.

> **Cómo leer este notebook**
>
> Antes de cada celda de código hay una **nota** como esta que explica, en palabras simples, qué es cada nombre que aparece (librerías, funciones, variables y términos técnicos). El código y sus resultados son los de la ejecución real del 17/10/2026.
>
> Los protagonistas, en una línea cada uno:
> - **Corpus**: el "manual" que puede consultar el agente (4 documentos sobre la API de GitHub).
> - **Embeddings + FAISS**: convierten cada pedazo del manual en números y permiten encontrar los que se parecen a la pregunta.
> - **BM25**: búsqueda por palabras clave; se combina con la anterior (búsqueda híbrida).
> - **Ollama + `qwen2.5:7b`**: el programa y el modelo de lenguaje que corren en la PC y redactan las respuestas.
> - **LangChain**: la librería que une todas las piezas.
> - **Agente + herramientas**: el modelo decide si busca en el manual, consulta GitHub en vivo o usa la calculadora de cuota.

## 1. Configuración
Todas las decisiones de diseño (modelos, tamaño de chunk, cantidad de fragmentos recuperados) se centralizan aquí. El modelo del LLM puede cambiarse con la variable de entorno `OLLAMA_MODEL` sin tocar el código.

> **Nota para entender el código — qué es cada nombre de la celda de abajo**
>
> - **`import ...`**: trae librerías (código ya hecho por otros) para usarlas acá. `json`, `os`, `re`, `math`, `time`, `unicodedata`, `Counter`, `datetime` y `Path` vienen con Python; sirven para leer archivos, buscar patrones en textos, hacer cuentas, medir tiempos, etc.
> - **`numpy` (`np`)**: librería para hacer cuentas con listas de números (vectores) de forma rápida.
> - **`requests`**: librería para pedirle datos a internet; la usamos para hablar con la API de GitHub.
> - **`SentenceTransformer`**: la clase de la librería *sentence-transformers* que carga un modelo que convierte textos en vectores (embeddings).
> - **`warnings.filterwarnings`**: silencia avisos de "esta librería va a cambiar" que no afectan el funcionamiento.
> - **LangChain**: el "kit de LEGO" para armar sistemas con LLMs. De ahí vienen:
>   - `create_agent`: arma el agente (modelo + herramientas + instrucciones).
>   - `FAISS` y `DistanceStrategy`: la base de datos de vectores y la forma de medir cuán parecidos son.
>   - `Embeddings`: el "molde" que tiene que respetar cualquier conversor de texto a vectores.
>   - `AIMessage` y `ToolMessage`: los tipos de mensaje de la conversación interna del agente (lo que dice el modelo y lo que devuelve una herramienta).
>   - `ChatPromptTemplate`, `RunnableLambda`, `RunnablePassthrough` y `StrOutputParser`: piezas para armar una "línea de montaje" de pasos (se explican en la sección 7.1).
>   - `tool`: la etiqueta que convierte una función en una herramienta que el agente puede usar.
>   - `ChatOllama`: el conector que le habla a Ollama, el programa que corre el modelo en la PC.
>   - `MarkdownHeaderTextSplitter` y `RecursiveCharacterTextSplitter`: las "tijeras" que cortan el manual en pedazos.
> - **Constantes en MAYÚSCULAS**: valores de configuración que se definen una sola vez.
>   - `CORPUS_DIR`: carpeta del manual. `INDICE_DIR`: carpeta donde se guarda el índice FAISS.
>   - `MODELO_EMBEDDINGS`: el modelo que pasa texto a números. `MODELO_EMBEDDINGS_BASE`: el modelo viejo, sólo para comparar.
>   - `CHUNK_SIZE`: tamaño máximo de cada pedazo (en letras). `CHUNK_OVERLAP`: cuántas letras se repiten entre pedazos si hay que cortar una sección.
>   - `TOP_K`: cuántos pedazos recibe el modelo por pregunta. `CANDIDATOS`: cuántos aporta cada buscador antes de combinar. `RRF_K`: constante de la fórmula que combina los dos buscadores.
>   - `MODELO_LLM`: el modelo de lenguaje (`qwen2.5:7b`; "7b" = 7 mil millones de parámetros). `OLLAMA_URL`: la dirección local donde escucha Ollama. `CONTEXTO_LLM`: cuánto texto puede "tener en la cabeza" el modelo a la vez, medido en *tokens* (pedacitos de palabra).
> - **`os.environ.get(...)`**: lee una variable de entorno (una configuración de Windows); si no existe, usa el valor por defecto.

In [1]:
# Dependencias (si faltan): %pip install -r requirements.txt
import json
import math
import os
import re
import sys
import time
import unicodedata
import warnings
from collections import Counter
from datetime import datetime
from pathlib import Path

import numpy as np
import requests
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore", category=DeprecationWarning)  # avisos de deprecación de langchain-community

from langchain.agents import create_agent
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.embeddings import Embeddings
from langchain_core.messages import AIMessage, ToolMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

# --- Rutas ---
CORPUS_DIR = Path("corpus")
INDICE_DIR = Path("faiss_github_api_index")

# --- RAG ---
MODELO_EMBEDDINGS = "intfloat/multilingual-e5-small"              # modelo usado por el sistema
MODELO_EMBEDDINGS_BASE = "sentence-transformers/all-MiniLM-L6-v2"  # sólo para la comparación de la sección 6
CHUNK_SIZE = 1500      # caracteres máximos por fragmento (ninguna sección del corpus lo supera)
CHUNK_OVERLAP = 150    # solapamiento si una sección larga debe subdividirse
TOP_K = 4              # fragmentos que recibe el LLM por consulta
CANDIDATOS = 10        # candidatos que aporta cada buscador (semántico y BM25) antes de fusionar
RRF_K = 60             # constante de Reciprocal Rank Fusion

# --- LLM local (Ollama) ---
MODELO_LLM = os.environ.get("OLLAMA_MODEL", "qwen2.5:7b")
OLLAMA_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
CONTEXTO_LLM = 8192    # tokens de contexto: el valor por defecto de Ollama es menor y truncaría los fragmentos

print(f"Python {sys.version.split()[0]} | LLM: {MODELO_LLM} | Embeddings: {MODELO_EMBEDDINGS}")

C:\Users\valet.DESKTOP-L670R7P\Desktop\IA_TP2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python 3.12.10 | LLM: qwen2.5:7b | Embeddings: intfloat/multilingual-e5-small


## 2. Fase 2 — Construcción del corpus y limpieza
El corpus está formado por cuatro documentos técnicos redactados a partir de la documentación oficial de GitHub (docs.github.com/rest), estructurados con encabezados Markdown por endpoint:

1. `01_autenticacion_y_rate_limits.md` — URL base, cabeceras, tipos de token, límites de uso y códigos HTTP.
2. `02_repositorios_y_ramas.md` — repositorios, ramas y *releases*.
3. `03_issues_y_etiquetas.md` — issues, comentarios, búsqueda y etiquetas.
4. `04_pull_requests_y_revisiones.md` — pull requests, *merge* y revisiones.

**Limpieza:** se normalizan saltos de línea, se eliminan separadores horizontales (`---`), marcas de negrita y líneas vacías repetidas. Los encabezados, tablas y bloques de código se conservan porque contienen la información técnica exacta (rutas, parámetros, JSON).

> **Nota para entender el código — qué es cada nombre de la celda de abajo**
>
> - **`def`**: así se define una función en Python (una receta con nombre que se puede reutilizar).
> - **`limpiar_markdown`**: función nuestra que limpia el texto de un documento antes de cortarlo.
> - **`re.sub`**: "buscar y reemplazar" usando patrones (expresiones regulares). Por ejemplo, `^\s*---\s*$` significa "una línea que sólo tiene tres guiones".
> - **Markdown (`.md`)**: el formato de los documentos del corpus; los títulos se marcan con `#`, `##`, `###` y las negritas con `**`.
> - **`archivos`**: la lista de los 4 documentos del manual (se excluye el README, que sólo describe la carpeta).
> - **`corpus`**: un diccionario (pares nombre → contenido) con el texto ya limpio de cada documento.
> - **`read_text(encoding="utf-8")`**: lee el archivo respetando tildes y eñes.

In [2]:
def limpiar_markdown(texto: str) -> str:
    # Normaliza el texto de un documento antes de segmentarlo.
    texto = texto.replace("\r\n", "\n")
    texto = re.sub(r"^\s*---\s*$", "", texto, flags=re.MULTILINE)   # separadores horizontales
    texto = texto.replace("**", "")                                  # negritas (ruido para la búsqueda)
    texto = re.sub(r"[ \t]+$", "", texto, flags=re.MULTILINE)        # espacios al final de línea
    texto = re.sub(r"\n{3,}", "\n\n", texto)                          # líneas vacías repetidas
    return texto.strip()


archivos = sorted(p for p in CORPUS_DIR.glob("*.md") if not p.name.lower().startswith("readme"))
corpus = {}
for ruta in archivos:
    original = ruta.read_text(encoding="utf-8")
    corpus[ruta.name] = limpiar_markdown(original)
    print(f"{ruta.name:40s} {len(original):6,d} -> {len(corpus[ruta.name]):6,d} caracteres")

01_autenticacion_y_rate_limits.md         6,081 ->  5,960 caracteres
02_repositorios_y_ramas.md                4,709 ->  4,559 caracteres
03_issues_y_etiquetas.md                  5,497 ->  5,341 caracteres
04_pull_requests_y_revisiones.md          4,783 ->  4,681 caracteres


## 3. Fase 2 — Segmentación (*chunking*)
Se usa una estrategia en dos etapas:

1. **`MarkdownHeaderTextSplitter`**: corta el documento en las secciones definidas por los encabezados `#`, `##` y `###`. Así cada fragmento corresponde a un endpoint o tema y nunca mezcla, por ejemplo, "crear un issue" con "crear una etiqueta".
2. **`RecursiveCharacterTextSplitter`**: si una sección supera `CHUNK_SIZE` (1500 caracteres), la subdivide priorizando párrafos, ítems de lista y líneas, con `CHUNK_OVERLAP` caracteres de solapamiento.

**Por qué 1500 caracteres:** en una primera versión se usaron 800, y las secciones largas quedaban partidas. Por ejemplo, la tabla de cabeceras HTTP se cortaba entre filas y el agente sólo veía la fila de `User-Agent`. Con 1500 caracteres ninguna sección actual se subdivide, así que cada fragmento es un endpoint completo (tablas y listas de parámetros incluidas). El divisor recursivo queda como resguardo para secciones más largas. 1500 caracteres equivalen a unos 400 tokens, dentro del límite de 512 del modelo de embeddings.

Además, a cada fragmento se le antepone la **ruta de encabezados** (documento › sección › subsección). Un fragmento que sólo dice "*Parámetros requeridos: `title`*" no indica de qué endpoint habla; con el encabezado contextual sí, y eso mejora tanto la búsqueda como la respuesta del LLM.

> **Nota para entender el código — qué es cada nombre de la celda de abajo**
>
> - **`ENCABEZADOS`**: le dice a la tijera qué símbolos son títulos y qué nombre ponerles (`documento`, `seccion`, `subseccion`).
> - **`MarkdownHeaderTextSplitter` (`divisor_markdown`)**: corta el texto en cada título. Resultado: una sección por endpoint, por ejemplo "1.2 Crear un Issue".
> - **`RecursiveCharacterTextSplitter` (`divisor_recursivo`)**: tijera de respaldo; si una sección pasa de `CHUNK_SIZE` letras, la corta probando primero entre párrafos (`\n\n`), después entre ítems de lista (`\n* `), después entre líneas.
> - **`ruta_encabezados`**: arma la "dirección" del pedazo, como `Issues y Etiquetas > 1. Endpoints para Issues > 1.2 Crear un Issue`.
> - **`segmentar_corpus`**: aplica las dos tijeras a cada documento y a cada pedazo le pega su dirección al principio, para que no pierda contexto.
> - **`chunks`**: la lista final de pedazos (*chunks*). Cada uno es un `Document` de LangChain con dos partes: `page_content` (el texto) y `metadata` (datos extra: archivo de origen, títulos, número de pedazo).
> - **`np.mean`**: el promedio (acá, el largo medio de los pedazos).

In [3]:
ENCABEZADOS = [("#", "documento"), ("##", "seccion"), ("###", "subseccion")]
divisor_markdown = MarkdownHeaderTextSplitter(headers_to_split_on=ENCABEZADOS, strip_headers=True)
divisor_recursivo = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n* ", "\n", " "],
)


def ruta_encabezados(metadatos: dict) -> str:
    return " > ".join(metadatos[k] for k in ("documento", "seccion", "subseccion") if k in metadatos)


def segmentar_corpus(corpus: dict) -> list:
    chunks = []
    for nombre, texto in corpus.items():
        secciones = divisor_markdown.split_text(texto)
        for seccion in secciones:
            seccion.metadata["source"] = nombre
        for fragmento in divisor_recursivo.split_documents(secciones):
            fragmento.metadata["id"] = len(chunks)
            fragmento.metadata["ruta"] = ruta_encabezados(fragmento.metadata)
            fragmento.page_content = f"[{fragmento.metadata['ruta']}]\n{fragmento.page_content}"
            chunks.append(fragmento)
    return chunks


chunks = segmentar_corpus(corpus)
longitudes = [len(c.page_content) for c in chunks]
print(f"Fragmentos generados: {len(chunks)} (secciones del corpus: {sum(len(divisor_markdown.split_text(t)) for t in corpus.values())})")
print(f"Longitud (caracteres): mín {min(longitudes)} | media {np.mean(longitudes):.0f} | máx {max(longitudes)}")
for nombre in corpus:
    print(f"  {nombre:40s} {sum(c.metadata['source'] == nombre for c in chunks):3d} fragmentos")

ejemplo = next(c for c in chunks if "Crear un Issue" in c.metadata["ruta"])
print("\nEjemplo de fragmento:\n" + "-" * 70 + f"\n{ejemplo.page_content}\n" + "-" * 70)

Fragmentos generados: 39 (secciones del corpus: 39)
Longitud (caracteres): mín 150 | media 562 | máx 1429
  01_autenticacion_y_rate_limits.md         12 fragmentos
  02_repositorios_y_ramas.md                 8 fragmentos
  03_issues_y_etiquetas.md                  11 fragmentos
  04_pull_requests_y_revisiones.md           8 fragmentos

Ejemplo de fragmento:
----------------------------------------------------------------------
[Issues y Etiquetas > 1. Endpoints para Issues (Incidencias) > 1.2 Crear un Issue]
* Método y Ruta: `POST /repos/{owner}/{repo}/issues`
* Código de respuesta exitosa: `201 Created`
* Cuerpo de la Solicitud (JSON Payload):
```json
{
"title": "Falla de autenticación en endpoint /user/repos",
"body": "Al enviar un token expirado, el mensaje devuelto debe ser 401 en lugar de 403.",
"assignees": ["octocat"],
"milestone": 1,
"labels": ["bug", "backend", "prioridad-alta"]
}
```
* Parámetros Requeridos (campos obligatorios):
* `title` (string): Título breve y descriptiv

## 4. Fase 2 — Embeddings
Un *embedding* es un vector que representa el significado de un texto: textos con significado parecido quedan cerca en el espacio vectorial.

Se eligió **`intfloat/multilingual-e5-small`** porque el corpus y las preguntas están en **español**; el modelo que usaba la versión anterior (`all-MiniLM-L6-v2`) fue entrenado principalmente en inglés (en la sección 6 se comparan ambos). Los modelos E5 se entrenaron con prefijos: `"query: "` para preguntas y `"passage: "` para documentos. La clase siguiente adapta `sentence-transformers` a la interfaz `Embeddings` de LangChain y agrega esos prefijos. Los vectores se **normalizan** (norma 1), de modo que el producto interno equivale a la similitud coseno.

> **Nota para entender el código — qué es cada nombre de la celda de abajo**
>
> - **Embedding**: una lista de números (acá, 384) que representa el significado de un texto. Textos que dicen cosas parecidas quedan con números parecidos, como puntos cercanos en un mapa.
> - **`class`**: define un "tipo de objeto" propio, con datos y funciones adentro.
> - **`EmbeddingsSentenceTransformers`**: clase nuestra que hace de adaptador (como un adaptador de enchufe) entre la librería *sentence-transformers* y LangChain.
> - **`embed_documents` / `embed_query`**: las dos funciones que LangChain exige: una convierte los pedazos del manual y la otra convierte la pregunta.
> - **`encode`**: la función del modelo que hace la conversión de texto a números.
> - **`normalize_embeddings=True`**: deja todos los vectores con "largo" 1, así sólo importa hacia dónde apuntan. Eso permite medir el parecido con la *similitud coseno*: 1 = significan lo mismo, 0 = nada que ver.
> - **`batch_size=16`**: procesa de a 16 textos por vez (más rápido que de a uno).
> - **`"query: "` y `"passage: "`**: etiquetas que el modelo e5 necesita delante de las preguntas y de los textos del manual, porque así fue entrenado.
> - **`intfloat/multilingual-e5-small`**: el modelo elegido. *multilingual* = entiende español; *small* = versión liviana; *intfloat* es la cuenta que lo publicó en Hugging Face (la "tienda" gratuita de modelos).
> - **`np.linalg.norm`**: calcula el "largo" de un vector (debería dar 1.000).

In [4]:
class EmbeddingsSentenceTransformers(Embeddings):
    # Adaptador entre un modelo de sentence-transformers y LangChain.

    def __init__(self, nombre_modelo: str, prefijo_consulta: str = "", prefijo_documento: str = ""):
        self.nombre_modelo = nombre_modelo
        self.modelo = SentenceTransformer(nombre_modelo)
        self.prefijo_consulta = prefijo_consulta
        self.prefijo_documento = prefijo_documento

    def embed_documents(self, textos: list) -> list:
        textos = [self.prefijo_documento + t for t in textos]
        return self.modelo.encode(textos, normalize_embeddings=True, batch_size=16).tolist()

    def embed_query(self, texto: str) -> list:
        return self.modelo.encode(self.prefijo_consulta + texto, normalize_embeddings=True).tolist()


embeddings = EmbeddingsSentenceTransformers(MODELO_EMBEDDINGS, prefijo_consulta="query: ", prefijo_documento="passage: ")
vector = np.array(embeddings.embed_query("¿Cómo creo un issue?"))
print(f"Dimensión de los vectores: {vector.size} | norma: {np.linalg.norm(vector):.3f}")
print(f"Primeros 6 componentes: {np.round(vector[:6], 4)}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  40%|████      | 80/199 [00:00<00:00, 679.01it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1300.35it/s]

Dimensión de los vectores: 384 | norma: 1.000
Primeros 6 componentes: [ 0.0772 -0.0355 -0.0848 -0.0866  0.0579 -0.1162]


## 5. Fase 2 — Base vectorial (FAISS) y búsqueda híbrida
**FAISS** (Facebook AI Similarity Search) almacena los vectores y resuelve búsquedas de vecinos más cercanos. Con `DistanceStrategy.MAX_INNER_PRODUCT` se crea un índice `IndexFlatIP` (búsqueda exacta por producto interno); como los vectores están normalizados, el puntaje es la **similitud coseno**. Para un corpus de este tamaño la búsqueda exacta es instantánea y no requiere índices aproximados. El índice se guarda en disco para poder reutilizarlo.

La documentación técnica tiene muchos términos exactos (`title`, `422`, `User-Agent`, "obligatorio") que una búsqueda puramente semántica puede pasar por alto con un modelo de embeddings pequeño. Por eso se combina con una búsqueda **léxica BM25** (ranking clásico por frecuencia de términos, ponderado por su rareza en el corpus y normalizado por la longitud del fragmento). Cada buscador aporta sus 10 mejores candidatos y se fusionan con **Reciprocal Rank Fusion (RRF)**:

$$\text{RRF}(d) = \sum_{b \in \{\text{semántico},\ \text{BM25}\}} \frac{1}{60 + \text{rango}_b(d)}$$

RRF sólo usa las posiciones, no los puntajes, por lo que no hace falta calibrar escalas distintas (coseno vs. BM25). Un fragmento bien ubicado en ambas listas sube al principio.

> **Nota para entender el código — qué es cada nombre de la celda de abajo**
>
> - **FAISS**: la base de datos de vectores (la hizo Facebook). Guarda los vectores de todos los pedazos y encuentra rápido los más parecidos a la pregunta.
> - **`FAISS.from_documents`**: convierte todos los pedazos en vectores y los guarda en el índice.
> - **`DistanceStrategy.MAX_INNER_PRODUCT` / `IndexFlatIP`**: la forma de comparar. *Flat* = compara contra todos, sin atajos (búsqueda exacta); *IP* = producto interno, que con vectores normalizados es la similitud coseno.
> - **`save_local`**: guarda el índice en una carpeta para poder reutilizarlo. **`index.ntotal`**: cuántos vectores tiene.
> - **`vectorstore`**: como llama LangChain a esta base de vectores. **`similarity_search`**: "dame los k pedazos más parecidos a este texto".
> - **BM25**: búsqueda clásica por palabras clave (la de los buscadores de antes de la IA). Da más puntaje si el pedazo tiene las palabras de la pregunta, sobre todo las poco comunes, y no favorece a los pedazos largos.
> - **`STOPWORDS`**: palabras que no ayudan a buscar ("el", "de", "que"…) y se ignoran.
> - **`tokenizar`**: corta un texto en palabras sueltas, en minúscula, sin tildes (`unicodedata` separa la letra de su tilde y después la tilde se descarta) y sin el plural final (*issues* → *issue*).
> - **`IndiceBM25`**: nuestra implementación de BM25. Adentro: `idf` = cuán rara es cada palabra en el corpus; `k1` y `b` = perillas estándar de la fórmula; `largo_medio` = largo promedio de los pedazos; `Counter` = cuenta cuántas veces aparece cada palabra.
> - **`recuperar`**: la búsqueda **híbrida**. Pide 10 candidatos a FAISS (por significado) y 10 a BM25 (por palabras) y los combina con **RRF** (*Reciprocal Rank Fusion*): cada pedazo suma `1 / (60 + puesto)` por cada lista en la que aparece. El que quedó arriba en las dos, gana. Devuelve los `TOP_K` mejores.
> - **`buscar`**: sólo para mostrar en pantalla qué encontró la búsqueda y en qué puesto quedó cada pedazo en cada buscador.

In [5]:
# --- Búsqueda semántica: FAISS ---
vectorstore = FAISS.from_documents(chunks, embeddings, distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)
vectorstore.save_local(str(INDICE_DIR))
print(f"Índice {type(vectorstore.index).__name__} con {vectorstore.index.ntotal} vectores guardado en '{INDICE_DIR}/'")

# --- Búsqueda léxica: BM25 ---
STOPWORDS = set(
    "a al algo como con cual cuales cuando de del el ella en entre es esta este esto hay la las lo los mas me mi "
    "mis muy no o para pero por que se si sin sobre su sus un una uno unos y ya yo tu te ti le les nos son ser "
    "fue era donde quiero puedo debo hace hacer cada otro otra".split()
)


def tokenizar(texto: str) -> list:
    # Minúsculas, sin tildes, sin stopwords y con un recorte simple de plurales (issues -> issue).
    texto = unicodedata.normalize("NFD", texto.lower())
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    tokens = re.findall(r"[a-z0-9_]+", texto)
    return [t[:-1] if len(t) > 4 and t.endswith("s") else t for t in tokens if t not in STOPWORDS]


class IndiceBM25:
    # Implementación de Okapi BM25 sobre los mismos fragmentos indexados en FAISS.

    def __init__(self, documentos: list, k1: float = 1.5, b: float = 0.75):
        self.documentos = documentos
        self.tokens = [tokenizar(d.page_content) for d in documentos]
        self.frecuencias = [Counter(t) for t in self.tokens]
        self.k1, self.b = k1, b
        self.largo_medio = sum(map(len, self.tokens)) / len(self.tokens)
        n = len(documentos)
        df = Counter(t for toks in self.tokens for t in set(toks))
        self.idf = {t: math.log(1 + (n - f + 0.5) / (f + 0.5)) for t, f in df.items()}

    def puntajes(self, consulta: str) -> list:
        terminos = tokenizar(consulta)
        resultado = []
        for frec, toks in zip(self.frecuencias, self.tokens):
            norma = self.k1 * (1 - self.b + self.b * len(toks) / self.largo_medio)
            resultado.append(sum(self.idf[t] * frec[t] * (self.k1 + 1) / (frec[t] + norma) for t in terminos if t in frec))
        return resultado

    def buscar(self, consulta: str, k: int) -> list:
        p = self.puntajes(consulta)
        orden = sorted(range(len(p)), key=lambda i: p[i], reverse=True)[:k]
        return [self.documentos[i] for i in orden if p[i] > 0]


indice_bm25 = IndiceBM25(chunks)


# --- Fusión: Reciprocal Rank Fusion ---
def recuperar(consulta: str, k: int = TOP_K, detalle: bool = False) -> list:
    # Devuelve los k fragmentos con mayor puntaje RRF combinando ambos buscadores.
    listas = {
        "semántico": vectorstore.similarity_search(consulta, k=CANDIDATOS),
        "BM25": indice_bm25.buscar(consulta, k=CANDIDATOS),
    }
    puntaje, documento, rangos = {}, {}, {}
    for buscador, resultados in listas.items():
        for rango, doc in enumerate(resultados, 1):
            clave = doc.metadata["id"]
            puntaje[clave] = puntaje.get(clave, 0.0) + 1 / (RRF_K + rango)
            documento[clave] = doc
            rangos.setdefault(clave, {})[buscador] = rango
    mejores = sorted(puntaje, key=puntaje.get, reverse=True)[:k]
    if detalle:
        return [(documento[c], puntaje[c], rangos[c]) for c in mejores]
    return [documento[c] for c in mejores]


def buscar(consulta: str) -> None:
    # Muestra los fragmentos recuperados y su posición en cada buscador.
    print(f"\nConsulta: {consulta}")
    for i, (doc, rrf, rangos) in enumerate(recuperar(consulta, detalle=True), 1):
        posiciones = ", ".join(f"{b} #{r}" for b, r in rangos.items())
        print(f"  {i}. [RRF {rrf:.4f} | {posiciones}] {doc.metadata['source']} :: {doc.metadata['ruta'].split(' > ', 1)[-1]}")


buscar("¿Qué cabeceras hay que enviar y qué pasa si falta el User-Agent?")
buscar("¿Qué campos son obligatorios para crear un issue?")
buscar("¿Qué métodos existen para fusionar un pull request?")

Índice IndexFlatIP con 39 vectores guardado en 'faiss_github_api_index/'

Consulta: ¿Qué cabeceras hay que enviar y qué pasa si falta el User-Agent?
  1. [RRF 0.0323 | semántico #3, BM25 #1] 01_autenticacion_y_rate_limits.md :: 2. Encabezados HTTP Estándar
  2. [RRF 0.0320 | semántico #1, BM25 #4] 01_autenticacion_y_rate_limits.md :: 4. Cuotas de Uso y Límites de Tasa (Rate Limiting) > 4.2 Encabezados de Respuesta para Monitoreo de Cuota
  3. [RRF 0.0320 | semántico #2, BM25 #3] 03_issues_y_etiquetas.md :: 1. Endpoints para Issues (Incidencias) > 1.2 Crear un Issue
  4. [RRF 0.0297 | semántico #10, BM25 #5] 02_repositorios_y_ramas.md :: 1. Endpoints para Repositorios > 1.1 Listar Repositorios

Consulta: ¿Qué campos son obligatorios para crear un issue?
  1. [RRF 0.0315 | semántico #6, BM25 #1] 03_issues_y_etiquetas.md :: 1. Endpoints para Issues (Incidencias) > 1.2 Crear un Issue
  2. [RRF 0.0315 | semántico #2, BM25 #5] 03_issues_y_etiquetas.md :: 1. Endpoints para Issues (Incidencias

## 6. Evaluación del recuperador
Para justificar las decisiones se armó un conjunto de **16 preguntas** redactadas con palabras distintas a las de los encabezados (por ejemplo "abrir una incidencia" en lugar de "crear un issue"), cada una con la sección que debería recuperarse. Se miden, con `k = 4` (los fragmentos que efectivamente recibe el LLM):

* **Hit@1**: proporción de preguntas cuyo primer fragmento es de la sección correcta.
* **Hit@4**: proporción con la sección correcta entre los 4 primeros.
* **MRR** (*Mean Reciprocal Rank*): promedio de 1/posición del primer acierto (0 si no aparece).

Todas las configuraciones usan exactamente los mismos fragmentos, así que las diferencias se deben sólo al método de búsqueda.

> **Nota para entender el código — qué es cada nombre de la celda de abajo**
>
> - **`CONSULTAS_EVALUACION`**: 16 preguntas de prueba, cada una con la sección del manual que *debería* aparecer. Están redactadas con otras palabras a propósito ("abrir una incidencia" en vez de "crear un issue").
> - **`evaluar_recuperador`**: le hace las 16 preguntas a un buscador y cuenta aciertos.
> - **Hit@1**: en qué proporción de preguntas el **primer** resultado fue el correcto.
> - **Hit@4**: en qué proporción el correcto estuvo **entre los 4 primeros** (que es lo que recibe el modelo).
> - **MRR** (*Mean Reciprocal Rank*): un promedio que premia que el correcto aparezca bien arriba: vale 1 si salió primero, ½ si salió segundo, ⅓ si salió tercero, 0 si no apareció.
> - **`lambda q, k: ...`**: una función "de una línea" sin nombre; acá sirve para que los cuatro métodos de búsqueda se puedan probar con la misma función de evaluación.
> - **`embeddings_base` / `vectorstore_base`**: el modelo viejo (en inglés) y su índice, armados sólo para comparar.
> - **`del`**: borra esas variables al final para liberar memoria.

In [6]:
CONSULTAS_EVALUACION = [
    ("¿Qué ocurre si mi programa no manda el User-Agent?", "Encabezados HTTP"),
    ("¿Cuántas peticiones por hora se permiten sin token?", "Tipos de Límites"),
    ("¿Cómo veo cuánta cuota me queda sin gastar solicitudes?", "Endpoint Dedicado"),
    ("¿Qué significa recibir un error 422?", "Códigos de Estado"),
    ("Quiero crear un repo nuevo con licencia MIT y un .gitignore de Python", "Crear un Repositorio"),
    ("¿Qué permiso necesita el token para borrar un repositorio?", "Eliminar Repositorios"),
    ("¿Cómo publico la versión v1.0.0 de mi proyecto?", "Crear un Release"),
    ("¿Qué dato es obligatorio para abrir una incidencia nueva?", "Crear un Issue"),
    ("¿Cómo marco un issue como descartado al cerrarlo?", "Cerrar un Issue"),
    ("¿En qué formato se indica el color de una etiqueta?", "Crear una Nueva Etiqueta"),
    ("¿Qué estrategias hay para integrar los cambios de un PR en main?", "Fusionar un Pull Request"),
    ("¿Cómo apruebo formalmente los cambios de un compañero?", "Crear una Revisión"),
    ("¿Cómo sé si un pull request tiene conflictos antes de fusionarlo?", "Obtener un Pull Request"),
    ("¿Cómo filtro las incidencias por etiqueta y fecha de modificación?", "Listar Issues"),
    ("¿Qué diferencia hay entre un token clásico y uno de grano fino?", "Personal Access Tokens"),
    ("¿Cómo cuento sólo los issues abiertos sin contar los pull requests?", "Contar Issues Abiertos"),
]


def evaluar_recuperador(buscar_k, consultas, k=TOP_K) -> dict:
    # buscar_k(consulta, k) debe devolver una lista ordenada de documentos.
    aciertos_1 = aciertos_k = suma_rr = 0
    for consulta, seccion_esperada in consultas:
        resultados = buscar_k(consulta, k)
        posiciones = [i for i, d in enumerate(resultados, 1) if seccion_esperada.lower() in d.metadata["ruta"].lower()]
        if posiciones:
            aciertos_k += 1
            aciertos_1 += posiciones[0] == 1
            suma_rr += 1 / posiciones[0]
    n = len(consultas)
    return {"Hit@1": aciertos_1 / n, f"Hit@{k}": aciertos_k / n, "MRR": suma_rr / n}


embeddings_base = EmbeddingsSentenceTransformers(MODELO_EMBEDDINGS_BASE)
vectorstore_base = FAISS.from_documents(chunks, embeddings_base, distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)

configuraciones = {
    "Semántica all-MiniLM-L6-v2 (inglés)": lambda q, k: vectorstore_base.similarity_search(q, k=k),
    "Semántica multilingual-e5-small": lambda q, k: vectorstore.similarity_search(q, k=k),
    "Léxica BM25": lambda q, k: indice_bm25.buscar(q, k),
    "Híbrida e5 + BM25 (RRF) [elegida]": lambda q, k: recuperar(q, k),
}
evaluacion = {nombre: evaluar_recuperador(f, CONSULTAS_EVALUACION) for nombre, f in configuraciones.items()}

print(f"{'Método de búsqueda':38s} {'Hit@1':>7s} {'Hit@4':>7s} {'MRR':>7s}")
for nombre, m in evaluacion.items():
    print(f"{nombre:38s} {m['Hit@1']:7.2f} {m['Hit@4']:7.2f} {m['MRR']:7.2f}")
del vectorstore_base, embeddings_base  # el modelo en inglés sólo se usa para comparar

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1045.79it/s]

Método de búsqueda                       Hit@1   Hit@4     MRR
Semántica all-MiniLM-L6-v2 (inglés)       0.38    0.62    0.48
Semántica multilingual-e5-small           0.69    0.94    0.78
Léxica BM25                               0.75    0.94    0.83
Híbrida e5 + BM25 (RRF) [elegida]         0.88    0.94    0.89


## 7. Fase 3 — Conexión e integración con el LLM local (Ollama)
**Ollama** ejecuta el modelo de lenguaje en la propia computadora y lo expone como un servicio HTTP en `http://localhost:11434`. LangChain se comunica con él mediante `ChatOllama`. Se usa **temperatura 0** para obtener respuestas deterministas (importante al elegir herramientas) y se amplía la ventana de contexto a `CONTEXTO_LLM` tokens para que entren las instrucciones, los fragmentos recuperados y los resultados de las herramientas.

> **Nota para entender el código — qué es cada nombre de la celda de abajo**
>
> - **Ollama**: el programa que corre el modelo de lenguaje en la propia PC y lo deja disponible en `http://localhost:11434` (*localhost* = "esta misma computadora").
> - **`ChatOllama`**: el conector de LangChain para hablar con Ollama. Sus opciones:
>   - `model`: qué modelo usar (`qwen2.5:7b`).
>   - `base_url`: dónde está Ollama.
>   - `temperature=0`: la "creatividad" en cero; siempre elige la respuesta más probable, sin improvisar.
>   - `num_ctx`: cuánto texto entra en la "memoria de trabajo" del modelo (8192 tokens ≈ 5000 palabras).
> - **`llm.invoke(...)`**: le manda un mensaje al modelo y espera la respuesta. **`.content`**: el texto de esa respuesta.
> - **`time.perf_counter()`**: un cronómetro para medir cuánto tardó.

In [7]:
llm = ChatOllama(model=MODELO_LLM, base_url=OLLAMA_URL, temperature=0, num_ctx=CONTEXTO_LLM)

inicio = time.perf_counter()
prueba = llm.invoke("Responde únicamente con la palabra: conectado")
print(f"Respuesta de {MODELO_LLM}: {prueba.content!r} ({time.perf_counter() - inicio:.1f} s, incluye la carga del modelo en memoria)")

Respuesta de qwen2.5:7b: 'conectado' (40.8 s, incluye la carga del modelo en memoria)


### 7.1 Cadena RAG con LCEL
Antes de construir el agente se valida el RAG "puro" con una cadena de LangChain Expression Language:

```
pregunta ──► recuperar (híbrida, top-4) ──► formatear_contexto ──┐
    │                                                           ├──► prompt ──► LLM ──► respuesta
    └───────────────────────────────────────────────────────────┘
```

El *prompt* obliga al modelo a usar sólo el contexto, citar el archivo de origen y admitir cuando la información no está en el corpus (regla anti-alucinación).

> **Nota para entender el código — qué es cada nombre de la celda de abajo**
>
> - **`formatear_contexto`**: arma un solo texto con los pedazos encontrados, numerados y con su archivo de origen (así el modelo puede citar la fuente).
> - **Prompt**: el mensaje completo que se le manda al modelo. **`ChatPromptTemplate.from_template`**: crea un "molde" de prompt con huecos (`{contexto}` y `{pregunta}`) que se completan en cada consulta.
> - **`PROMPT_RAG`**: nuestro molde, con las reglas: usar sólo el contexto, citar la fuente y admitir cuando no sabe.
> - **LCEL** (*LangChain Expression Language*): la forma de LangChain de encadenar pasos con el símbolo `|`, como una línea de montaje donde la salida de un paso es la entrada del siguiente.
> - **`cadena_rag`**: la línea de montaje completa: pregunta → buscar pedazos → armar el mensaje → modelo → texto de respuesta.
>   - `RunnableLambda(recuperar)`: mete nuestra función de búsqueda en la cadena.
>   - `RunnablePassthrough()`: deja pasar la pregunta tal cual.
>   - `StrOutputParser()`: de todo lo que devuelve el modelo, se queda sólo con el texto.
> - **`cadena_rag.invoke(pregunta)`**: ejecuta toda la cadena para una pregunta.

In [8]:
def formatear_contexto(documentos) -> str:
    # Une los fragmentos recuperados indicando su archivo de origen (trazabilidad).
    return "\n\n".join(
        f"--- FRAGMENTO {i} | FUENTE: {doc.metadata['source']} ---\n{doc.page_content}"
        for i, doc in enumerate(documentos, 1)
    )


PROMPT_RAG = ChatPromptTemplate.from_template(
    "Eres un asistente técnico especializado en la API REST de GitHub (versión 2022-11-28).\n"
    "Responde la pregunta usando ÚNICAMENTE la información del CONTEXTO.\n\n"
    "Reglas:\n"
    "1. Indica métodos HTTP, rutas, parámetros, cabeceras y códigos de estado tal como aparecen en el contexto.\n"
    "2. Cita el archivo de origen de cada dato, por ejemplo [Fuente: 03_issues_y_etiquetas.md].\n"
    "3. Si el contexto no contiene la respuesta, responde exactamente: "
    "\"La documentación del corpus no contiene información suficiente para responder esta consulta.\"\n"
    "4. Responde en español y de forma concisa.\n\n"
    "CONTEXTO:\n{contexto}\n\nPREGUNTA:\n{pregunta}"
)

cadena_rag = (
    {"contexto": RunnableLambda(recuperar) | formatear_contexto, "pregunta": RunnablePassthrough()}
    | PROMPT_RAG
    | llm
    | StrOutputParser()
)

for pregunta in [
    "¿Qué método HTTP y qué ruta se usan para crear un pull request, y cuáles son sus campos obligatorios?",
    "¿Cómo se configura un webhook en un repositorio?",  # fuera del corpus: debe reconocerlo
]:
    print(f"PREGUNTA: {pregunta}")
    inicio = time.perf_counter()
    respuesta = cadena_rag.invoke(pregunta)
    print(f"RESPUESTA ({time.perf_counter() - inicio:.1f} s):\n{respuesta}\n" + "=" * 90)

PREGUNTA: ¿Qué método HTTP y qué ruta se usan para crear un pull request, y cuáles son sus campos obligatorios?


RESPUESTA (36.5 s):
El método HTTP y la ruta para crear un pull request son:

- Método: `POST`
- Ruta: `/repos/{owner}/{repo}/pulls`

Los campos obligatorios son:

- `title` (string)
- `head` (string)
- `base` (string)

Fuente: 04_pull_requests_y_revisiones.md
PREGUNTA: ¿Cómo se configura un webhook en un repositorio?


RESPUESTA (19.1 s):
La documentación del corpus no contiene información suficiente para responder esta consulta.


## 8. Fase 3 — Herramientas del agente
El agente dispone de tres herramientas. El LLM decide cuáles usar leyendo su **nombre, descripción y parámetros** (el *docstring* de cada función):

| Herramienta | Tipo | Qué hace |
| :--- | :--- | :--- |
| `consultar_documentacion_github` | RAG | Recupera los 4 fragmentos más relevantes del corpus (búsqueda híbrida) con su fuente. |
| `consultar_repositorio_github` | Acción externa (búsqueda en vivo) | Llama a la API real de GitHub y devuelve estrellas, forks, issues y PRs abiertos, lenguaje, etc. |
| `calcular_cuota_rate_limit` | Cálculo | Calcula el consumo de la cuota horaria (60 o 5000 solicitudes/hora) para un caso dado. |

Detalles de implementación:
* El campo `open_issues_count` de `GET /repos/{owner}/{repo}` **suma issues y pull requests abiertos**. Para informar la cantidad real de issues, la herramienta usa la Search API (`GET /search/issues` con `is:issue is:open`), tal como describe el propio corpus.
* Las solicitudes envían las cabeceras recomendadas por la documentación y, si existe la variable de entorno `GITHUB_TOKEN`, se autentican para tener una cuota mayor. Si GitHub redirige el repositorio (por ejemplo, porque fue transferido a otra organización), la herramienta lo informa.
* Las salidas incluyen un recordatorio o un resumen en lenguaje natural: los modelos pequeños siguen mejor las instrucciones cuando aparecen junto a los datos.

> **Nota para entender el código — qué es cada nombre de la celda de abajo**
>
> - **API REST**: la forma de pedirle datos a GitHub usando direcciones web. **`GITHUB_API`**: la dirección base (`https://api.github.com`).
> - **`cabeceras_github`**: arma las *cabeceras*, los datos de identificación que viajan con cada pedido:
>   - `Accept` (qué formato de respuesta queremos), `X-GitHub-Api-Version` (qué versión de la API), `User-Agent` (quién pide; GitHub lo exige).
>   - `Authorization: Bearer ...`: el token de GitHub (una contraseña para programas). Sólo se usa si existe la variable `GITHUB_TOKEN`.
> - **`contar_resultados_busqueda`**: usa el buscador de GitHub (`GET /search/issues`) y devuelve `total_count`, la cantidad de resultados. Se usa para contar issues y pull requests **por separado**.
> - **`@tool`**: la etiqueta que convierte la función de abajo en una herramienta del agente. El texto entre comillas triples (el *docstring*) es lo que lee el modelo para decidir cuándo usarla; `Args` explica cada dato que hay que pasarle.
> - **`consultar_documentacion_github`**: herramienta 1 (RAG). Busca en el manual con `recuperar` y devuelve los pedazos con su fuente.
> - **`consultar_repositorio_github`**: herramienta 2 (en vivo). Pide a GitHub los datos actuales de un repositorio:
>   - `owner` = dueño (ej. `facebook`); `repo` = nombre (ej. `react`).
>   - `r.status_code`: el código de respuesta (200 = todo bien; 404 = no existe; 403 o 429 = se pasó el límite de pedidos).
>   - `stargazers_count` (estrellas), `forks_count` (copias del proyecto), `full_name` (nombre real, que puede cambiar si GitHub lo redirige).
>   - `open_issues_count`: GitHub suma ahí issues **y** pull requests; por eso se usa el buscador para separarlos.
>   - **issue** = un "ticket" (un error o una tarea). **Pull request (PR)** = una propuesta de cambios al código.
>   - `json.dumps`: convierte el resultado a texto con formato JSON para que el modelo lo lea.
> - **`calcular_cuota_rate_limit`**: herramienta 3 (calculadora). El *rate limit* es el máximo de pedidos por hora (60 sin token, 5000 con token). Calcula el porcentaje usado, cuántos sobran o se pasan y cada cuántos segundos se puede pedir.
> - **`herramientas`**: la lista de las tres herramientas. **`.invoke({...})`**: prueba cada herramienta a mano, sin el modelo.

In [9]:
GITHUB_API = "https://api.github.com"


def cabeceras_github() -> dict:
    # Cabeceras recomendadas en 01_autenticacion_y_rate_limits.md.
    cabeceras = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "IA-TP2-Agente-RAG",
    }
    if os.environ.get("GITHUB_TOKEN"):
        cabeceras["Authorization"] = f"Bearer {os.environ['GITHUB_TOKEN']}"
    return cabeceras


def contar_resultados_busqueda(consulta: str):
    # Devuelve total_count de GET /search/issues, o None si la búsqueda falla.
    try:
        r = requests.get(f"{GITHUB_API}/search/issues", params={"q": consulta, "per_page": 1},
                         headers=cabeceras_github(), timeout=15)
        if r.status_code == 200:
            return r.json().get("total_count")
    except requests.RequestException:
        pass
    return None


@tool
def consultar_documentacion_github(consulta: str) -> str:
    '''Busca en la documentación técnica de la API REST de GitHub y devuelve los fragmentos más
    relevantes con su archivo de origen. Úsala para preguntas sobre endpoints, métodos HTTP,
    parámetros, cabeceras, códigos de estado, autenticación o límites de uso (rate limit).

    Args:
        consulta: tema o pregunta técnica a buscar, en español (ej. "campos obligatorios para crear un issue").
    '''
    documentos = recuperar(consulta)
    if not documentos:
        return "No se encontraron fragmentos relevantes en la documentación."
    return ("Fragmentos recuperados de la documentación. Usa sólo lo que dicen estos fragmentos; si no "
            "contienen la respuesta, indica que la documentación del corpus no incluye esa información.\n\n"
            + formatear_contexto(documentos))


@tool
def consultar_repositorio_github(owner: str, repo: str) -> str:
    '''Consulta EN VIVO la API pública de GitHub y devuelve datos actuales de un repositorio:
    estrellas, forks, issues abiertos (sin contar pull requests), pull requests abiertos,
    lenguaje principal, descripción y fecha del último push.

    Args:
        owner: usuario u organización dueña del repositorio (ej. "facebook").
        repo: nombre del repositorio (ej. "react").
    '''
    if "/" in owner:  # tolera entradas del tipo owner="facebook/react"
        owner, _, resto = owner.partition("/")
        repo = repo or resto
    owner, repo = owner.strip(), repo.strip()
    try:
        r = requests.get(f"{GITHUB_API}/repos/{owner}/{repo}", headers=cabeceras_github(), timeout=15)
    except requests.RequestException as error:
        return f"Error de conexión con la API de GitHub: {error}"
    if r.status_code == 404:
        return f"Error 404: el repositorio '{owner}/{repo}' no existe o es privado."
    if r.status_code in (403, 429):
        return (f"Error {r.status_code}: se alcanzó el rate limit de la API de GitHub "
                f"(x-ratelimit-remaining={r.headers.get('x-ratelimit-remaining')}). Reintentar más tarde.")
    if r.status_code != 200:
        return f"Error HTTP {r.status_code} al consultar la API de GitHub."

    datos = r.json()
    nombre = datos["full_name"]
    issues = contar_resultados_busqueda(f"repo:{nombre} is:issue is:open")
    pull_requests = contar_resultados_busqueda(f"repo:{nombre} is:pr is:open")
    resultado = {"fuente": "datos obtenidos en vivo de la API de GitHub", "repositorio": nombre}
    if nombre.lower() != f"{owner}/{repo}".lower():
        resultado["nota"] = f"GitHub redirigió '{owner}/{repo}' a '{nombre}' (el repositorio fue transferido o renombrado)."
    resultado.update({
        "descripcion": datos.get("description") or "Sin descripción",
        "lenguaje_principal": datos.get("language") or "No especificado",
        "estrellas": datos.get("stargazers_count"),
        "forks": datos.get("forks_count"),
    })
    if issues is not None and pull_requests is not None:
        resultado["issues_abiertos"] = issues
        resultado["pull_requests_abiertos"] = pull_requests
    else:  # la Search API falló: se informa el total combinado aclarando qué incluye
        resultado["issues_y_pull_requests_abiertos"] = datos.get("open_issues_count")
    resultado.update({
        "ultimo_push": datos.get("pushed_at"),
        "consultado": datetime.now().astimezone().strftime("%Y-%m-%d %H:%M (hora local)"),
        "url": datos.get("html_url"),
    })
    return json.dumps(resultado, ensure_ascii=False, indent=2)


@tool
def calcular_cuota_rate_limit(solicitudes_por_hora: int, autenticado: bool = False) -> str:
    '''Calcula el consumo de la cuota horaria de la API REST de GitHub (60 solicitudes por hora sin
    autenticar, 5000 con token) para una cantidad de solicitudes por hora. Devuelve el límite, las
    solicitudes restantes o excedentes, el porcentaje consumido y el intervalo mínimo entre solicitudes.

    Args:
        solicitudes_por_hora: cantidad de solicitudes que se planea hacer en una hora.
        autenticado: True si se usa un token (PAT, OAuth o GitHub App); False si son anónimas.
    '''
    limite = 5000 if autenticado else 60
    tipo = "con token" if autenticado else "sin autenticar"
    porcentaje = 100 * solicitudes_por_hora / limite
    excedente = max(0, solicitudes_por_hora - limite)
    restantes = max(0, limite - solicitudes_por_hora)
    if excedente:
        resumen = (f"{solicitudes_por_hora} solicitudes por hora {tipo} SUPERAN el límite de {limite} por hora "
                   f"({porcentaje:.1f}% de la cuota, {excedente} solicitudes de exceso).")
    else:
        resumen = (f"{solicitudes_por_hora} solicitudes por hora {tipo} están DENTRO del límite de {limite} por hora "
                   f"({porcentaje:.1f}% de la cuota; quedan {restantes} disponibles).")
    return json.dumps({
        "resumen": resumen,
        "tipo_de_acceso": tipo,
        "limite_por_hora": limite,
        "solicitudes_planificadas": solicitudes_por_hora,
        "porcentaje_de_cuota": f"{porcentaje:.1f}%",
        "solicitudes_restantes": restantes,
        "solicitudes_excedentes": excedente,
        "intervalo_minimo_entre_solicitudes_seg": round(3600 / limite, 2),
    }, ensure_ascii=False, indent=2)


herramientas = [consultar_documentacion_github, consultar_repositorio_github, calcular_cuota_rate_limit]

# Prueba directa de cada herramienta (sin LLM)
print(consultar_documentacion_github.invoke({"consulta": "límite de solicitudes por hora"})[:500], "...\n")
print(consultar_repositorio_github.invoke({"owner": "octocat", "repo": "Hello-World"}), "\n")
print(calcular_cuota_rate_limit.invoke({"solicitudes_por_hora": 80, "autenticado": False}))

Fragmentos recuperados de la documentación. Usa sólo lo que dicen estos fragmentos; si no contienen la respuesta, indica que la documentación del corpus no incluye esa información.

--- FRAGMENTO 1 | FUENTE: 01_autenticacion_y_rate_limits.md ---
[Autenticación, Configuración y Rate Limiting > 4. Cuotas de Uso y Límites de Tasa (Rate Limiting) > 4.1 Tipos de Límites Principales]
1. Peticiones no autenticadas:
* Límite: 60 solicitudes por hora por dirección IP de origen.
* Aplica a consultas públi ...



{
  "fuente": "datos obtenidos en vivo de la API de GitHub",
  "repositorio": "octocat/Hello-World",
  "descripcion": "My first repository on GitHub!",
  "lenguaje_principal": "No especificado",
  "estrellas": 3850,
  "forks": 7120,
  "issues_abiertos": 6130,
  "pull_requests_abiertos": 1315,
  "ultimo_push": "2024-08-20T23:54:42Z",
  "consultado": "2026-10-17 22:39 (hora local)",
  "url": "https://github.com/octocat/Hello-World"
} 

{
  "resumen": "80 solicitudes por hora sin autenticar SUPERAN el límite de 60 por hora (133.3% de la cuota, 20 solicitudes de exceso).",
  "tipo_de_acceso": "sin autenticar",
  "limite_por_hora": 60,
  "solicitudes_planificadas": 80,
  "porcentaje_de_cuota": "133.3%",
  "solicitudes_restantes": 0,
  "solicitudes_excedentes": 20,
  "intervalo_minimo_entre_solicitudes_seg": 60.0
}


## 9. Fase 3 — Orquestación del agente (ciclo ReAct)
`create_agent` de LangChain construye un agente que repite el ciclo **ReAct** (*Reasoning + Acting*):

1. **Razonar:** el LLM recibe la pregunta, las instrucciones del sistema y la descripción de las herramientas, y decide si necesita alguna.
2. **Actuar:** si la necesita, responde con una *tool call* (nombre de la herramienta + argumentos en JSON). El LLM no ejecuta nada: LangChain ejecuta la función de Python.
3. **Observar:** el resultado vuelve al LLM como un mensaje de herramienta.
4. Se repite hasta que el LLM tiene la información suficiente y redacta la **respuesta final** integrada.

La función `ejecutar_consulta` muestra cada acción y observación para que el flujo sea trazable.

> **Nota para entender el código — qué es cada nombre de la celda de abajo**
>
> - **Agente**: un modelo de lenguaje que, además de responder, puede decidir usar herramientas.
> - **`PROMPT_SISTEMA`**: las instrucciones fijas que recibe el modelo antes de cada pregunta (qué herramientas tiene, cuándo usar cada una, citar la fuente y no inventar).
> - **`create_agent`**: la función de LangChain que arma el agente juntando `model` (el LLM), `tools` (las herramientas) y `system_prompt` (las instrucciones).
> - **ReAct**: el ciclo que sigue el agente: **Ra**zonar → **Act**uar (usar una herramienta) → observar el resultado → repetir o responder.
> - **Tool calling**: cuando el modelo, en vez de responder, contesta "quiero usar tal herramienta con tales datos". El modelo no ejecuta nada; LangChain ejecuta la función de Python y le devuelve el resultado.
> - **`ejecutar_consulta`**: nuestra función para hacerle una pregunta al agente y mostrar paso a paso qué hizo:
>   - `agente.invoke({"messages": [...]})`: le pasa la pregunta como un mensaje con `role: "user"` (el usuario).
>   - `recursion_limit=12`: un freno para que el agente no dé vueltas para siempre.
>   - `estado["messages"]`: toda la conversación interna. Los `AIMessage` con `tool_calls` son los pedidos de herramienta (se muestran como **[Acción]**) y los `ToolMessage` son los resultados (se muestran como **[Observación]**). El último mensaje es la respuesta final.
> - **`registro_interacciones`**: lista donde se guarda cada prueba, para después grabarla en un archivo.

In [10]:
PROMPT_SISTEMA = '''Eres un asistente técnico experto en la API REST de GitHub. Dispones de tres herramientas:
- consultar_documentacion_github: busca en la documentación técnica del corpus (endpoints, métodos HTTP, parámetros, cabeceras, códigos de estado, autenticación y límites de uso).
- consultar_repositorio_github: obtiene datos EN VIVO de un repositorio público concreto (estrellas, forks, issues y pull requests abiertos, lenguaje).
- calcular_cuota_rate_limit: calcula el consumo de la cuota horaria para una cantidad de solicitudes por hora.

Cómo trabajar:
1. Identifica todas las partes de la pregunta y llama a TODAS las herramientas necesarias antes de responder. Si la pregunta de documentación abarca varios temas, haz una búsqueda por tema.
2. Si la pregunta menciona endpoints, métodos HTTP, parámetros o campos, cabeceras, códigos de estado o de error, autenticación o límites, DEBES llamar a consultar_documentacion_github, aunque también uses otras herramientas. Nunca respondas esos datos de memoria.
3. Si el usuario nombra un repositorio (por ejemplo facebook/react) y pide datos actuales, usa consultar_repositorio_github.
4. Si el usuario menciona una cantidad de solicitudes por hora, usa calcular_cuota_rate_limit una vez por cada escenario e informa el límite que corresponde y el porcentaje consumido.

Reglas para la respuesta final:
- Escribe en español, de forma breve y ordenada, respondiendo cada parte de la pregunta en el mismo orden.
- Usa EXCLUSIVAMENTE la información devuelta por las herramientas; nunca la completes con conocimiento propio.
- Sé completo: si los fragmentos enumeran varios elementos (cabeceras, campos, códigos, opciones), menciónalos todos, y distingue claramente los campos obligatorios de los opcionales.
- Si los fragmentos de la documentación no mencionan lo que se pregunta (por ejemplo, un endpoint que no aparece en ellos), responde que la documentación del corpus no contiene esa información.
- Agrega [Fuente: archivo.md] después de cada dato tomado de la documentación. Presenta los resultados de calcular_cuota_rate_limit como un cálculo y los de consultar_repositorio_github como datos obtenidos en vivo de la API de GitHub.'''

agente = create_agent(model=llm, tools=herramientas, system_prompt=PROMPT_SISTEMA)
registro_interacciones = []


def ejecutar_consulta(pregunta: str) -> dict:
    # Ejecuta el agente sobre una pregunta y muestra la traza de acciones y observaciones.
    print("=" * 90 + f"\nUSUARIO: {pregunta}\n" + "=" * 90)
    inicio = time.perf_counter()
    estado = agente.invoke({"messages": [{"role": "user", "content": pregunta}]}, config={"recursion_limit": 12})
    segundos = time.perf_counter() - inicio

    acciones, observaciones = [], []
    for mensaje in estado["messages"]:
        if isinstance(mensaje, AIMessage) and mensaje.tool_calls:
            for llamada in mensaje.tool_calls:
                acciones.append({"herramienta": llamada["name"], "argumentos": llamada["args"]})
                print(f"  [Acción]      {llamada['name']}({json.dumps(llamada['args'], ensure_ascii=False)})")
        elif isinstance(mensaje, ToolMessage):
            observaciones.append({"herramienta": mensaje.name, "resultado": mensaje.content})
            print(f"  [Observación] {mensaje.name}: {mensaje.content[:150].replace(chr(10), ' ')}...")

    respuesta = estado["messages"][-1].content
    print(f"\nAGENTE ({segundos:.1f} s):\n{respuesta}\n")
    resultado = {"pregunta": pregunta, "acciones": acciones, "observaciones": observaciones,
                 "respuesta": respuesta, "segundos": round(segundos, 1)}
    registro_interacciones.append(resultado)
    return resultado


print(f"Agente listo con {len(herramientas)} herramientas: {[h.name for h in herramientas]}")

Agente listo con 3 herramientas: ['consultar_documentacion_github', 'consultar_repositorio_github', 'calcular_cuota_rate_limit']


## 10. Pruebas del flujo completo
Se evalúan cinco consultas que ejercitan todas las combinaciones del flujo esperado:

| Caso | Pregunta | Herramientas esperadas |
| :--- | :--- | :--- |
| 1 | Cabeceras y error por falta de `User-Agent` | Documentación (RAG) |
| 2 | ¿80 solicitudes/hora superan el límite? ¿Y con token? | Documentación + cálculo |
| 3 | Métricas actuales de `facebook/react` | Consulta en vivo |
| 4 | Cómo crear un issue + cuántos issues abiertos tiene `facebook/react` | Documentación + consulta en vivo |
| 5 | Pregunta fuera del corpus (webhooks) | Documentación → debe admitir que no tiene la información |

> **Nota para entender el código — qué es cada nombre de la celda de abajo**
>
> - Cada celda de abajo le hace **una** pregunta al agente con `ejecutar_consulta`. En la salida vas a ver:
>   - **[Acción]**: qué herramienta decidió usar el agente y con qué datos.
>   - **[Observación]**: el comienzo de lo que devolvió esa herramienta.
>   - **AGENTE (x s)**: la respuesta final y cuántos segundos tardó (el modelo corre en el procesador, sin placa de video).
> - **[Fuente: archivo.md]**: el agente indica de qué documento del manual sacó cada dato.
> - **`;`** al final de la línea: evita que Jupyter imprima de nuevo el diccionario de resultados que devuelve la función.

In [11]:
# Caso 1: sólo RAG
ejecutar_consulta("¿Qué cabeceras HTTP se recomienda enviar en cada solicitud a la API REST de GitHub "
                  "y qué ocurre si una petición no incluye el User-Agent?");

USUARIO: ¿Qué cabeceras HTTP se recomienda enviar en cada solicitud a la API REST de GitHub y qué ocurre si una petición no incluye el User-Agent?


  [Acción]      consultar_documentacion_github({"consulta": "cabeceras HTTP recomendadas"})
  [Observación] consultar_documentacion_github: Fragmentos recuperados de la documentación. Usa sólo lo que dicen estos fragmentos; si no contienen la respuesta, indica que la documentación del corp...

AGENTE (86.4 s):
En cada solicitud a la API REST de GitHub se recomienda incluir los siguientes encabezados:

- `Accept`: `application/vnd.github+json` (Especifica la versión del tipo de medio oficial de GitHub para respuestas JSON).
- `Authorization`: `Bearer <TOKEN_PERSONAL>` (Proporciona las credenciales de acceso para peticiones autenticadas).
- `X-GitHub-Api-Version`: `2022-11-28` (Fija la versión de la especificación de la API para evitar incompatibilidades).
- `User-Agent`: `<NOMBRE_APLICACION_O_USUARIO>` (Obligatorio por política de GitHub. Peticiones sin `User-Agent` son rechazadas con error HTTP 403).

Si una petición no incluye el encabezado `User-Agent`, la solicitud será rechazada co

In [12]:
# Caso 2: RAG + herramienta de cálculo
ejecutar_consulta("Mi script hace 80 solicitudes por hora a la API de GitHub sin token. ¿Supera el límite? "
                  "¿Y si uso un token personal? ¿Qué código de error devolvería GitHub al excederlo?");

USUARIO: Mi script hace 80 solicitudes por hora a la API de GitHub sin token. ¿Supera el límite? ¿Y si uso un token personal? ¿Qué código de error devolvería GitHub al excederlo?


  [Acción]      calcular_cuota_rate_limit({"solicitudes_por_hora": 80, "autenticado": false})
  [Acción]      calcular_cuota_rate_limit({"autenticado": true, "solicitudes_por_hora": 80})
  [Acción]      consultar_documentacion_github({"consulta": "códigos de error de la API de GitHub"})
  [Observación] calcular_cuota_rate_limit: {   "resumen": "80 solicitudes por hora sin autenticar SUPERAN el límite de 60 por hora (133.3% de la cuota, 20 solicitudes de exceso).",   "tipo_de_a...
  [Observación] calcular_cuota_rate_limit: {   "resumen": "80 solicitudes por hora con token están DENTRO del límite de 5000 por hora (1.6% de la cuota; quedan 4920 disponibles).",   "tipo_de_a...
  [Observación] consultar_documentacion_github: Fragmentos recuperados de la documentación. Usa sólo lo que dicen estos fragmentos; si no contienen la respuesta, indica que la documentación del corp...

AGENTE (82.3 s):
La API de GitHub sin token recibe 80 solicitudes por hora, lo que supera el límite de 60 solicitud

In [13]:
# Caso 3: sólo consulta en vivo
ejecutar_consulta("¿Cuántas estrellas, forks, issues abiertos y pull requests abiertos tiene actualmente "
                  "el repositorio facebook/react, y cuál es su lenguaje principal?");

USUARIO: ¿Cuántas estrellas, forks, issues abiertos y pull requests abiertos tiene actualmente el repositorio facebook/react, y cuál es su lenguaje principal?


  [Acción]      consultar_repositorio_github({"owner": "facebook", "repo": "react"})
  [Observación] consultar_repositorio_github: {   "fuente": "datos obtenidos en vivo de la API de GitHub",   "repositorio": "react/react",   "nota": "GitHub redirigió 'facebook/react' a 'react/rea...

AGENTE (22.7 s):
El repositorio actualmente tiene 251,223 estrellas, 51,516 forks, 880 issues abiertos y 579 pull requests abiertos. Su lenguaje principal es JavaScript. [Fuente: datos obtenidos en vivo de la API de GitHub]



In [14]:
# Caso 4: flujo combinado RAG + consulta en vivo
ejecutar_consulta("¿Qué método HTTP y qué endpoint se usan para crear un issue, qué campo es obligatorio, "
                  "y cuántos issues abiertos tiene hoy el repositorio facebook/react?");

USUARIO: ¿Qué método HTTP y qué endpoint se usan para crear un issue, qué campo es obligatorio, y cuántos issues abiertos tiene hoy el repositorio facebook/react?


  [Acción]      consultar_documentacion_github({"consulta": "método HTTP y endpoint para crear un issue, campo obligatorio"})
  [Acción]      consultar_repositorio_github({"owner": "facebook", "repo": "react"})
  [Observación] consultar_documentacion_github: Fragmentos recuperados de la documentación. Usa sólo lo que dicen estos fragmentos; si no contienen la respuesta, indica que la documentación del corp...
  [Observación] consultar_repositorio_github: {   "fuente": "datos obtenidos en vivo de la API de GitHub",   "repositorio": "react/react",   "nota": "GitHub redirigió 'facebook/react' a 'react/rea...

AGENTE (56.2 s):
Para crear un issue, se utiliza el método HTTP `POST` y el endpoint `/repos/{owner}/{repo}/issues`. Los campos obligatorios son `title` (título breve y descriptivo del issue). [Fuente: 03_issues_y_etiquetas.md]

El repositorio facebook/react tiene actualmente 880 issues abiertos. [Fuente: datos obtenidos en vivo de la API de GitHub]



In [15]:
# Caso 5: pregunta fuera del dominio del corpus (control de alucinaciones)
ejecutar_consulta("Según la documentación, ¿qué endpoint se usa para crear un webhook en un repositorio?");

USUARIO: Según la documentación, ¿qué endpoint se usa para crear un webhook en un repositorio?


  [Acción]      consultar_documentacion_github({"consulta": "endpoint para crear un webhook en un repositorio"})
  [Observación] consultar_documentacion_github: Fragmentos recuperados de la documentación. Usa sólo lo que dicen estos fragmentos; si no contienen la respuesta, indica que la documentación del corp...

AGENTE (33.0 s):
La documentación del corpus no contiene información sobre un endpoint específico para crear un webhook en un repositorio.



## 11. Resumen de la ejecución

> **Nota para entender el código — qué es cada nombre de la celda de abajo**
>
> - Esta celda arma una tabla con las herramientas que usó el agente en cada caso y el tiempo de respuesta.
> - **`json.dump`**: guarda todo (modelo usado, evaluación del buscador y las 5 interacciones completas) en **`resultados_interaccion.json`**, un archivo de texto que después se usó para escribir el informe.

In [16]:
print(f"{'Caso':5s} {'Herramientas usadas':75s} {'Tiempo':>8s}")
for i, r in enumerate(registro_interacciones, 1):
    usadas = ", ".join(a["herramienta"] for a in r["acciones"]) or "(ninguna)"
    print(f"{i:<5d} {usadas:75s} {r['segundos']:7.1f}s")

with open("resultados_interaccion.json", "w", encoding="utf-8") as archivo:
    json.dump({"modelo_llm": MODELO_LLM, "modelo_embeddings": MODELO_EMBEDDINGS,
               "evaluacion_recuperador": evaluacion, "interacciones": registro_interacciones},
              archivo, ensure_ascii=False, indent=2)
print("\nResultados guardados en resultados_interaccion.json")

Caso  Herramientas usadas                                                           Tiempo
1     consultar_documentacion_github                                                 86.4s
2     calcular_cuota_rate_limit, calcular_cuota_rate_limit, consultar_documentacion_github    82.3s
3     consultar_repositorio_github                                                   22.7s
4     consultar_documentacion_github, consultar_repositorio_github                   56.2s
5     consultar_documentacion_github                                                 33.0s

Resultados guardados en resultados_interaccion.json
